In [2]:
import pandas as pd
#Encoding incorrecto
df = pd.read_csv("listings_1.csv", encoding="latin-1")
print(df.shape)
print(df.dtypes)
print(df.head())
print(df.isnull().sum())

(30468, 21)
id                                float64
name                               object
host_id                           float64
hosts_time_as_host_years            int64
neighbourhood_cleansed             object
latitude                          float64
longitude                         float64
room_type                          object
accommodates                        int64
bedrooms                          float64
beds                              float64
price                              object
minimum_nights                    float64
maximum_nights                    float64
availability_365                    int64
number_of_reviews                   int64
estimated_occupancy_l365d           int64
review_scores_rating              float64
calculated_host_listings_count      int64
Unnamed: 19                       float64
Unnamed: 20                       float64
dtype: object
         id                                               name   host_id  \
0  281134.0     

In [3]:
!pip install chardet
import chardet

with open("listings_1.csv", "rb") as f:
    resultado = chardet.detect(f.read(100000))
    print(resultado)

{'encoding': 'utf-8', 'confidence': 0.99, 'language': ''}


In [4]:
df = pd.read_csv("listings_1.csv", encoding="utf-8", encoding_errors="replace")

# Contar cuántos caracteres raros hay
total_raros = 0
for col in df.select_dtypes(include="object").columns:
    cuenta = df[col].astype(str).str.contains("�", na=False).sum()
    if cuenta > 0:
        print(f"{col}: {cuenta} valores con caracteres corruptos")
        total_raros += cuenta

print(f"\nTotal: {total_raros}")

name: 2 valores con caracteres corruptos

Total: 2


In [5]:
import pandas as pd
df = pd.read_csv("listings_1.csv", encoding="utf-8", encoding_errors="replace")
print(f"Forma: {df.shape}")
print(f"\nBarrios (primeros 5):")
print(df["neighbourhood_cleansed"].head().tolist())

Forma: (30468, 21)

Barrios (primeros 5):
['Cuauhtémoc', 'Cuajimalpa de Morelos', 'Cuauhtémoc', 'Cuauhtémoc', 'Coyoacán']


# Problema: Codificación mixta
- El archivo está en UTF-8 (99% confianza según chardet).
- Sin embargo, contiene 2 valores que impiden la carga directa.
- Se solucionó con encoding="utf-8", encoding_errors="replace".
- 2 filas afectadas, insignificante para el análisis.

In [6]:
# Quitar columnas vacías
print("ANTES")
print(f"Columnas: {df.shape[1]}")

df = df.drop(columns=["Unnamed: 19", "Unnamed: 20"], errors="ignore")

print(f"DESPUÉS")
print(f"Columnas: {df.shape[1]}")
print(f"Columnas finales: {df.columns.tolist()}")

# Cambiar la columna price a float64

print("\nPRICE: ANTES")
print(f"Tipo: {df['price'].dtype}")
print(f"Ejemplos: {df['price'].head(3).tolist()}")
print(f"Nulos: {df['price'].isnull().sum()}")

# Limpiar y convertir

df["price"] = df["price"].str.replace("$", "", regex=False)
df["price"] = df["price"].str.replace(",", "", regex=False)
df["price"] = pd.to_numeric(df["price"], errors="coerce")

print("\nPRICE: DESPUÉS")
print(f"Tipo: {df['price'].dtype}")
print(f"Ejemplos: {df['price'].head(3).tolist()}")
print(f"Nulos: {df['price'].isnull().sum()}")
print(f"Min: {df['price'].min()}")
print(f"Max: {df['price'].max()}")
print(f"Mediana: {df['price'].median()}")


# Eliminar filas sin precio

filas_antes = df.shape[0]
df = df.dropna(subset=["price"]).reset_index(drop=True)
filas_despues = df.shape[0]

print(f"\nELIMINACIÓN DE FILAS SIN PRECIO")
print(f"Antes: {filas_antes}")
print(f"Después: {filas_despues}")
print(f"Eliminadas: {filas_antes - filas_despues} ({(filas_antes-filas_despues)/filas_antes*100:.2f}%)")



print("\nhosts_time_as_host_years")
print(df["hosts_time_as_host_years"].value_counts())


# Resumen de nulos restantes

print("\nNULOS RESTANTES")
nulos = df.isnull().sum()
nulos_pct = (nulos / len(df) * 100).round(2)
resumen = pd.DataFrame({"Nulos": nulos, "Porcentaje": nulos_pct})
print(resumen[resumen["Nulos"] > 0].sort_values("Nulos", ascending=False))

ANTES
Columnas: 21
DESPUÉS
Columnas: 19
Columnas finales: ['id', 'name', 'host_id', 'hosts_time_as_host_years', 'neighbourhood_cleansed', 'latitude', 'longitude', 'room_type', 'accommodates', 'bedrooms', 'beds', 'price', 'minimum_nights', 'maximum_nights', 'availability_365', 'number_of_reviews', 'estimated_occupancy_l365d', 'review_scores_rating', 'calculated_host_listings_count']

PRICE: ANTES
Tipo: object
Ejemplos: ['$3,505.00', '$8,874.00', '$21,950.00']
Nulos: 2108

PRICE: DESPUÉS
Tipo: float64
Ejemplos: [3505.0, 8874.0, 21950.0]
Nulos: 2108
Min: 136.8
Max: 1826199.19
Mediana: 1500.0

ELIMINACIÓN DE FILAS SIN PRECIO
Antes: 30468
Después: 28360
Eliminadas: 2108 (6.92%)

hosts_time_as_host_years
hosts_time_as_host_years
0     4161
3     3108
2     2537
1     2533
4     2473
8     2310
7     2243
9     2075
6     2043
10    1743
5     1549
11     740
12     474
13     205
14     115
15      51
Name: count, dtype: int64

NULOS RESTANTES
                      Nulos  Porcentaje
review_s

In [7]:
print("PRICE")
print(f"Tipo: {df['price'].dtype}")
print(f"Nulos: {df['price'].isnull().sum()}")
print(f"Min: {df['price'].min()}")
print(f"Max: {df['price'].max()}")
print(f"Mediana: {df['price'].median()}")
print(f"Media: {df['price'].mean():.2f}")

# Duda con la columna hosts_time_as_host_years

print("\nhosts_time_as_host_years")
print(df["hosts_time_as_host_years"].value_counts().head(10))

# Cuáles valores siguen siendo nulos?

print("\nNULOS RESTANTES")
nulos = df.isnull().sum()
nulos_pct = (nulos / len(df) * 100).round(2)
resumen = pd.DataFrame({"Nulos": nulos, "Porcentaje": nulos_pct})
print(resumen[resumen["Nulos"] > 0].sort_values("Nulos", ascending=False))

# Distribuir precios (para detectar outliers)

print("\nDISTRIBUCIÓN DE PRECIOS (percentiles)")
print(df["price"].describe(percentiles=[.25, .5, .75, .90, .95, .99]))

PRICE
Tipo: float64
Nulos: 0
Min: 136.8
Max: 1826199.19
Mediana: 1500.0
Media: 2339.92

hosts_time_as_host_years
hosts_time_as_host_years
0     4161
3     3108
2     2537
1     2533
4     2473
8     2310
7     2243
9     2075
6     2043
10    1743
Name: count, dtype: int64

NULOS RESTANTES
                      Nulos  Porcentaje
review_scores_rating   3582       12.63
beds                    403        1.42
bedrooms                 23        0.08
minimum_nights            1        0.00
maximum_nights            1        0.00

DISTRIBUCIÓN DE PRECIOS (percentiles)
count    2.836000e+04
mean     2.339918e+03
std      1.409418e+04
min      1.368000e+02
25%      9.421975e+02
50%      1.500000e+03
75%      2.326625e+03
90%      3.756000e+03
95%      5.445637e+03
99%      1.347625e+04
max      1.826199e+06
Name: price, dtype: float64


# Columna price como texto ya está resulto
- Algunos precios venían como texto
- Al cargar con UTF-8, pandas interpretó los valores como numéricos directamente
- Ahorita son de tipo float64, no hay nulos
- No se requirió limpieza adicional

In [8]:
# Tratar outliers en la columna price
print("ANTES DE TRATAR OUTLIERS")
print(f"Filas: {len(df)}")
print(f"Max: {df['price'].max()}")

# Método IQR (Rango Intercuartílico)
Q1 = df["price"].quantile(0.25)
Q3 = df["price"].quantile(0.75)
IQR = Q3 - Q1
limite_superior = Q3 + 3 * IQR  # 3x IQR para ser conservador

print(f"\nQ1: {Q1}")
print(f"Q3: {Q3}")
print(f"IQR: {IQR}")
print(f"Límite superior (Q3 + 3*IQR): {limite_superior:.2f}")

# Registros que no son la norma
outliers = df[df["price"] > limite_superior]
print(f"\nOutliers detectados: {len(outliers)} ({len(outliers)/len(df)*100:.2f}%)")

# Ver algunos outliers
print("\nTOP 10 PRECIOS MÁS ALTOS")
print(df.nlargest(10, "price")[["name", "neighbourhood_cleansed", "room_type", "price"]])

ANTES DE TRATAR OUTLIERS
Filas: 28360
Max: 1826199.19

Q1: 942.1975
Q3: 2326.625
IQR: 1384.4275
Límite superior (Q3 + 3*IQR): 6479.91

Outliers detectados: 1059 (3.73%)

TOP 10 PRECIOS MÁS ALTOS
                                                    name  \
3900      Habitación con Servicio de Hotel en Roma Norte   
10587                  ¡Recámara acogedora y muy cómoda!   
21543            Beautiful apartment in Polanco, Mexico.   
9656                   Full house at heart of Roma Norte   
8052   The most luxurious and prívate stay in Mexico ...   
10740                                Majestuosa Hacienda   
6493     Departamento de 2 pisos en condesa con Roof top   
11621   4 Thoughtfully curated Villas / Heart of Condesa   
4289                Superb Suite few steps away Masaryk.   
10380                    Apartamento completo. Muy lindo   

      neighbourhood_cleansed        room_type       price  
3900              Cuauhtémoc     Private room  1826199.19  
10587                Tla

In [9]:
# Eliminar errores imposibles + capear en P99

print("ANTES")
print(f"Filas: {len(df)}")
print(f"Max: {df['price'].max():,.2f}")

# Eliminar precios imposibles (> $50,000 MXN)
UMBRAL_IMPOSIBLE = 50000
df = df[df["price"] <= UMBRAL_IMPOSIBLE].reset_index(drop=True)
print(f"\nEliminados > ${UMBRAL_IMPOSIBLE:,}: {28360 - len(df)} listings")

# Capear en P99 (los lujos reales se mantienen pero sin distorsionar)
P99 = df["price"].quantile(0.99)
print(f"\nP99 después de limpieza: ${P99:,.2f}")

# Guardar los que serán capeados
capeados = (df["price"] > P99).sum()
df["price"] = df["price"].clip(upper=P99)
print(f"Listings capeados a P99: {capeados}")

print(f"\nDESPUÉS")
print(f"Filas: {len(df)}")
print(f"Max: {df['price'].max():,.2f}")
print(f"Media: {df['price'].mean():,.2f}")
print(f"Mediana: {df['price'].median():,.2f}")

print("\nNUEVA DISTRIBUCIÓN")
print(df["price"].describe(percentiles=[.25, .5, .75, .90, .95, .99]))

ANTES
Filas: 28360
Max: 1,826,199.19

Eliminados > $50,000: 37 listings

P99 después de limpieza: $12,466.80
Listings capeados a P99: 284

DESPUÉS
Filas: 28323
Max: 12,466.80
Media: 2,014.23
Mediana: 1,500.00

NUEVA DISTRIBUCIÓN
count    28323.000000
mean      2014.231509
std       1885.387239
min        136.800000
25%        940.950000
50%       1500.000000
75%       2320.000000
90%       3734.950000
95%       5382.839000
99%      12456.504000
max      12466.800000
Name: price, dtype: float64


## Outliers en price

Se detectaron 1,059 listings (3.73%) por encima del límite Q3 + 3*IQR ($6,479.91).

Al inspeccionar el top 10, se identificaron valores imposibles  (ej. "Private room"
a $1,826,199 MXN/noche), que corresponden a errores de captura o rentas mensuales
publicadas como diarias.

1. Se eliminaron de listings con precio mayor a $50,000 MXN (errores evidentes).
2. Capeo al percentil 99 para valores de lujo legítimos que no deben distorsionar las medias.

- Fueron eliminados 37 listings
- Fueron capeados 284 listings  
- Mediana final: $1,500 MXN


In [10]:
# beds: imputar con mediana por tipo de habitación
print("BEDS: ANTES ")
print(df["beds"].isnull().sum())

df["beds"] = df.groupby("room_type")["beds"].transform(
    lambda x: x.fillna(x.median())
)
df["beds"] = df["beds"].astype("Int64")

print("BEDS: DESPUÉS")
print(df["beds"].isnull().sum())

# Crear columna de ingreso estimado (KPI)
df["ingreso_estimado"] = df["price"] * df["estimated_occupancy_l365d"]

# Reporte final
print("\nESTADO FINAL DEL DATASET")
print(f"Filas: {len(df)}")
print(f"Columnas: {df.shape[1]}")
print(f"\nNulos restantes:")
print(df.isnull().sum()[df.isnull().sum() > 0])

# Guardar dataset limpio
df.to_csv("listings_cdmx_clean.csv", index=False, encoding="utf-8")
print("\n✅ Guardado como 'listings_cdmx_clean.csv'")

from google.colab import files
files.download("listings_cdmx_clean.csv")

BEDS: ANTES 
401
BEDS: DESPUÉS
0

ESTADO FINAL DEL DATASET
Filas: 28323
Columnas: 20

Nulos restantes:
bedrooms                  23
minimum_nights             1
maximum_nights             1
review_scores_rating    3560
dtype: int64

✅ Guardado como 'listings_cdmx_clean.csv'


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [11]:
# KPI 1: Precio mediano por barrio

print("=" * 50)
print("KPI 1: PRECIO MEDIANO POR BARRIO")
print("=" * 50)

kpi_barrios = df.groupby("neighbourhood_cleansed").agg(
    listings=("id", "count"),
    precio_mediano=("price", "median"),
    precio_promedio=("price", "mean"),
    ocupacion_promedio=("estimated_occupancy_l365d", "mean"),
    ingreso_estimado_promedio=("ingreso_estimado", "mean")
).round(2).sort_values("precio_mediano", ascending=False)

print(kpi_barrios.head(15))

kpi_barrios.to_csv("kpi_barrios.csv")
print("\n✅ Guardado: kpi_barrios.csv")


# KPI 2: Precio por tipo de habitación

print("\n" + "=" * 50)
print("KPI 2: PRECIO POR TIPO DE HABITACIÓN")
print("=" * 50)

kpi_room = df.groupby("room_type").agg(
    listings=("id", "count"),
    precio_mediano=("price", "median"),
    precio_promedio=("price", "mean"),
    ocupacion_promedio=("estimated_occupancy_l365d", "mean"),
    ingreso_estimado_promedio=("ingreso_estimado", "mean")
).round(2).sort_values("precio_mediano", ascending=False)

print(kpi_room)
kpi_room.to_csv("kpi_room_type.csv")


# KPI 3: Top 20 listings por ingreso estimado

print("\n" + "=" * 50)
print("KPI 3: TOP 20 LISTINGS POR INGRESO ESTIMADO")
print("=" * 50)

top_ingresos = df.nlargest(20, "ingreso_estimado")[
    ["name", "neighbourhood_cleansed", "room_type", "price",
     "estimated_occupancy_l365d", "ingreso_estimado"]
]
print(top_ingresos.to_string(index=False))
top_ingresos.to_csv("top_ingresos.csv", index=False)


# KPI 4: Relación host nuevo vs. experimentado

print("\n" + "=" * 50)
print("KPI 4: ANTIGÜEDAD DEL HOST VS. PRECIO")
print("=" * 50)

df["tipo_host"] = pd.cut(
    df["hosts_time_as_host_years"],
    bins=[-1, 1, 3, 6, 100],
    labels=["Nuevo (0-1 año)", "Intermedio (2-3 años)", "Experto (4-6 años)", "Veterano (7+ años)"]
)

kpi_host = df.groupby("tipo_host", observed=True).agg(
    listings=("id", "count"),
    precio_mediano=("price", "median"),
    ocupacion_promedio=("estimated_occupancy_l365d", "mean"),
    rating_promedio=("review_scores_rating", "mean")
).round(2)

print(kpi_host)
kpi_host.to_csv("kpi_host.csv")


# KPI 5: Top 10 barrios

print("\n" + "=" * 50)
print("KPI 5: TOP 10 BARRIOS CON MÁS LISTINGS")
print("=" * 50)

top_barrios = df["neighbourhood_cleansed"].value_counts().head(10)
print(top_barrios)

KPI 1: PRECIO MEDIANO POR BARRIO
                        listings  precio_mediano  precio_promedio  \
neighbourhood_cleansed                                              
Miguel Hidalgo              4367         1879.00          2474.00   
Cuajimalpa de Morelos        363         1794.00          2257.73   
Cuauhtémoc                 13202         1699.12          2208.71   
Álvaro Obregón               918         1340.86          1770.97   
Coyoacán                    2055         1200.00          1752.40   
Benito Juárez               3202         1200.00          1575.62   
Iztacalco                    559         1159.00          1316.74   
Xochimilco                   188         1044.40          1868.30   
La Magdalena Contreras       117         1020.00          1661.84   
Tlalpan                      943         1014.67          1994.97   
Venustiano Carranza         1039          982.00          1233.38   
Azcapotzalco                 386          923.00          1190.44   
G

In [12]:
# Recargar los datos originales sin capping
df_original = pd.read_csv("listings_1.csv", encoding="utf-8", encoding_errors="replace")
df_original = df_original.drop(columns=["Unnamed: 19", "Unnamed: 20"], errors="ignore")
df_original["price"] = pd.to_numeric(df_original["price"].str.replace("$","",regex=False).str.replace(",","",regex=False), errors="coerce")
df_original = df_original.dropna(subset=["price"])
df_original = df_original[df_original["price"] <= 50000]
df_original["ingreso_estimado"] = df_original["price"] * df_original["estimated_occupancy_l365d"]

top_ingresos_real = df_original.nlargest(20, "ingreso_estimado")[
    ["name", "neighbourhood_cleansed", "room_type", "price",
     "estimated_occupancy_l365d", "ingreso_estimado"]
]
print(top_ingresos_real.to_string(index=False))
top_ingresos_real.to_csv("top_ingresos_real.csv",index=False)

                                                name neighbourhood_cleansed       room_type    price  estimated_occupancy_l365d  ingreso_estimado
                         Beauty apartment in La Roma             Cuauhtémoc Entire home/apt 37073.00                        216        8007768.00
  HUGE Decor Stylish Casona Condesa  SkywatchRooftop             Cuauhtémoc Entire home/apt 35775.99                        174        6225022.26
                     Departamento moderno en la ROMA             Cuauhtémoc Entire home/apt 23000.00                        255        5865000.00
             2BR Perfect Location Designer's Luxury.             Cuauhtémoc Entire home/apt 34227.67                        162        5544882.54
  Mexican Desgn Villa Pvt. Roofgardn (A/C ALL Bdrms)             Cuauhtémoc Entire home/apt 30444.40                        174        5297325.60
  Mexican Villa / AC +bath all 9 BDRMs / Pvt Rooftop             Cuauhtémoc Entire home/apt 33061.33                        